# L7: Lab - Product Scout

In this lab, you'll build a product comparison tool that runs on your phone. Over a month of shopping trips, you snap photos of products you see. CLIP (compiled for Snapdragon) embeds each photo, and Qdrant Edge stores everything on-device. Later, you can search "red shoes I saw last week" or "find something similar to this" across all your shopping history.

This is the product comparison use case from the Qdrant Context Hub: visual similarity search across time, running entirely on-device.

```
5 shopping trips, 50+ products
     |
See a product --> Snap photo --> CLIP (Snapdragon NPU)
                                       |
                              Qdrant Edge (on-device)
                                       |
            "How much were those headphones at Best Buy?"
```

## Setup

In [ ]:
!pip install qdrant-edge-py qai-hub "qai-hub-models[openai_clip]" torch Pillow matplotlib numpy

In [ ]:
import os
import sys
sys.path.append("..")

import torch
import clip
import numpy as np
import time
import datetime
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from qai_hub_models.models.openai_clip import Model as OpenAIClip
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)

clip_model = OpenAIClip.from_pretrained()

VECTOR_NAME = "product"
DIMENSION = 512

print("CLIP model loaded")

### Compile for Snapdragon

In [ ]:
import qai_hub
from utils import get_ai_hub_api_token, get_random_device

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

device_name = get_random_device()
device = qai_hub.Device(device_name)

class CLIPVisualEncoder(torch.nn.Module):
    def __init__(self, clip_visual):
        super().__init__()
        self.visual = clip_visual
    def forward(self, image):
        features = self.visual(image)
        return features / features.norm(dim=1, keepdim=True)

visual_encoder = CLIPVisualEncoder(clip_model.clip.visual)
traced = torch.jit.trace(visual_encoder, torch.randn(1, 3, 224, 224))

compile_job = qai_hub.submit_compile_job(
    model=traced,
    input_specs={"image": (1, 3, 224, 224)},
    device=device,
)
target_model = compile_job.get_target_model()
print(f"CLIP compiled for {device_name}")

## 1. Simulate Five Shopping Trips

We'll simulate products you photograph over a month of shopping. Each trip visits different stores, and products span shoes, clothing, electronics, accessories, and bags.

In [ ]:
PRODUCTS_DIR = "./product_photos"
Path(PRODUCTS_DIR).mkdir(parents=True, exist_ok=True)

np.random.seed(42)

def make_product_photo(name, base_color, noise_level=25):
    """Generate a synthetic product photo."""
    base = np.array(base_color, dtype=np.uint8)
    noise = np.random.randint(-noise_level, noise_level, (224, 224, 3), dtype=np.int16)
    frame = np.clip(base + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(frame)
    path = f"{PRODUCTS_DIR}/{name}.png"
    img.save(path)
    return path

# 5 shopping trips over a month
trips = {
    "trip1_saturday_mall": {
        "label": "Saturday at the mall",
        "day_offset": 28,  # 4 weeks ago
        "hour": 14,
        "products": [
            {"name": "red_sneakers_1", "color": [200, 50, 50], "category": "shoes", "store": "FootLocker", "price": 89, "desc": "Red running sneakers"},
            {"name": "blue_jacket_1", "color": [40, 60, 180], "category": "clothing", "store": "Nordstrom", "price": 120, "desc": "Navy blue winter jacket"},
            {"name": "silver_watch_1", "color": [180, 180, 190], "category": "accessories", "store": "Macy's", "price": 250, "desc": "Silver analog watch"},
            {"name": "green_backpack_1", "color": [50, 130, 50], "category": "bags", "store": "REI", "price": 75, "desc": "Green hiking backpack"},
            {"name": "black_headphones_1", "color": [30, 30, 35], "category": "electronics", "store": "Best Buy", "price": 299, "desc": "Black over-ear headphones"},
            {"name": "white_sneakers_1", "color": [240, 240, 235], "category": "shoes", "store": "Nike Store", "price": 110, "desc": "White leather sneakers"},
            {"name": "gray_tshirt_1", "color": [150, 150, 155], "category": "clothing", "store": "Uniqlo", "price": 25, "desc": "Gray cotton t-shirt"},
            {"name": "leather_belt_1", "color": [100, 60, 30], "category": "accessories", "store": "Nordstrom", "price": 65, "desc": "Brown leather belt"},
            {"name": "laptop_stand_1", "color": [170, 175, 180], "category": "electronics", "store": "Best Buy", "price": 45, "desc": "Aluminum laptop stand"},
            {"name": "canvas_tote_1", "color": [200, 190, 160], "category": "bags", "store": "Macy's", "price": 40, "desc": "Canvas tote bag"},
        ],
    },
    "trip2_wednesday_target": {
        "label": "Wednesday at Target and DSW",
        "day_offset": 21,  # 3 weeks ago
        "hour": 17,
        "products": [
            {"name": "red_boots_2", "color": [180, 40, 40], "category": "shoes", "store": "DSW", "price": 95, "desc": "Red ankle boots"},
            {"name": "gray_hoodie_2", "color": [120, 120, 125], "category": "clothing", "store": "Target", "price": 45, "desc": "Gray pullover hoodie"},
            {"name": "gold_watch_2", "color": [200, 170, 80], "category": "accessories", "store": "Target", "price": 60, "desc": "Gold-tone digital watch"},
            {"name": "blue_backpack_2", "color": [40, 80, 160], "category": "bags", "store": "Target", "price": 35, "desc": "Blue school backpack"},
            {"name": "wireless_earbuds_2", "color": [240, 240, 245], "category": "electronics", "store": "Best Buy", "price": 149, "desc": "White wireless earbuds"},
            {"name": "black_jeans_2", "color": [35, 35, 40], "category": "clothing", "store": "Target", "price": 40, "desc": "Black slim jeans"},
            {"name": "phone_case_2", "color": [60, 100, 180], "category": "accessories", "store": "Target", "price": 20, "desc": "Blue silicone phone case"},
            {"name": "running_shoes_2", "color": [100, 200, 100], "category": "shoes", "store": "DSW", "price": 75, "desc": "Green running shoes"},
            {"name": "usb_hub_2", "color": [50, 50, 55], "category": "electronics", "store": "Best Buy", "price": 30, "desc": "USB-C hub adapter"},
        ],
    },
    "trip3_sunday_outlets": {
        "label": "Sunday at the outlet mall",
        "day_offset": 14,  # 2 weeks ago
        "hour": 11,
        "products": [
            {"name": "tan_boots_3", "color": [180, 140, 90], "category": "shoes", "store": "Clarks Outlet", "price": 80, "desc": "Tan suede boots"},
            {"name": "puffer_jacket_3", "color": [20, 20, 25], "category": "clothing", "store": "North Face Outlet", "price": 150, "desc": "Black puffer jacket"},
            {"name": "smart_watch_3", "color": [40, 40, 45], "category": "accessories", "store": "Samsung Store", "price": 200, "desc": "Black smartwatch"},
            {"name": "messenger_bag_3", "color": [80, 60, 40], "category": "bags", "store": "Coach Outlet", "price": 130, "desc": "Brown leather messenger bag"},
            {"name": "bt_speaker_3", "color": [60, 130, 200], "category": "electronics", "store": "Bose Outlet", "price": 120, "desc": "Blue portable bluetooth speaker"},
            {"name": "white_dress_shirt_3", "color": [245, 245, 240], "category": "clothing", "store": "Brooks Brothers", "price": 55, "desc": "White dress shirt"},
            {"name": "sunglasses_3", "color": [30, 30, 30], "category": "accessories", "store": "Sunglass Hut", "price": 160, "desc": "Black aviator sunglasses"},
            {"name": "trail_shoes_3", "color": [130, 100, 70], "category": "shoes", "store": "Nike Outlet", "price": 65, "desc": "Brown trail running shoes"},
            {"name": "duffel_bag_3", "color": [60, 60, 65], "category": "bags", "store": "North Face Outlet", "price": 90, "desc": "Gray gym duffel bag"},
            {"name": "red_polo_3", "color": [190, 50, 50], "category": "clothing", "store": "Ralph Lauren", "price": 50, "desc": "Red polo shirt"},
        ],
    },
    "trip4_friday_electronics": {
        "label": "Friday at the electronics district",
        "day_offset": 7,  # 1 week ago
        "hour": 16,
        "products": [
            {"name": "noise_cancel_hp_4", "color": [35, 35, 40], "category": "electronics", "store": "Sony Store", "price": 350, "desc": "Black noise-canceling headphones"},
            {"name": "mechanical_kb_4", "color": [200, 200, 205], "category": "electronics", "store": "Best Buy", "price": 130, "desc": "White mechanical keyboard"},
            {"name": "webcam_4", "color": [25, 25, 30], "category": "electronics", "store": "Best Buy", "price": 80, "desc": "HD webcam with ring light"},
            {"name": "tablet_case_4", "color": [70, 70, 75], "category": "accessories", "store": "Apple Store", "price": 70, "desc": "Gray tablet folio case"},
            {"name": "charging_pad_4", "color": [240, 240, 240], "category": "electronics", "store": "Apple Store", "price": 40, "desc": "White wireless charging pad"},
            {"name": "earbuds_case_4", "color": [220, 200, 160], "category": "accessories", "store": "Amazon Pop-up", "price": 15, "desc": "Tan leather earbuds case"},
            {"name": "portable_monitor_4", "color": [20, 20, 25], "category": "electronics", "store": "Micro Center", "price": 200, "desc": "Portable USB-C monitor"},
            {"name": "cable_organizer_4", "color": [160, 160, 165], "category": "accessories", "store": "Micro Center", "price": 12, "desc": "Gray cable organizer pouch"},
        ],
    },
    "trip5_today_comparison": {
        "label": "Today: comparison shopping",
        "day_offset": 0,  # Today
        "hour": 13,
        "products": [
            {"name": "red_sneakers_5", "color": [210, 55, 45], "category": "shoes", "store": "Nike Store", "price": 95, "desc": "Red running sneakers (new model)"},
            {"name": "black_headphones_5", "color": [25, 25, 30], "category": "electronics", "store": "Target", "price": 249, "desc": "Black over-ear headphones (sale)"},
            {"name": "blue_jacket_5", "color": [35, 55, 170], "category": "clothing", "store": "REI", "price": 100, "desc": "Navy blue rain jacket"},
            {"name": "silver_watch_5", "color": [190, 190, 195], "category": "accessories", "store": "Amazon Pop-up", "price": 180, "desc": "Silver analog watch (similar)"},
            {"name": "green_backpack_5", "color": [55, 140, 55], "category": "bags", "store": "REI", "price": 65, "desc": "Green daypack"},
            {"name": "white_earbuds_5", "color": [245, 245, 248], "category": "electronics", "store": "Target", "price": 129, "desc": "White wireless earbuds (new gen)"},
            {"name": "brown_boots_5", "color": [140, 90, 50], "category": "shoes", "store": "DSW", "price": 85, "desc": "Brown Chelsea boots"},
        ],
    },
}

# Generate all product photos
all_products = []
today = datetime.date.today()

for trip_key, trip in trips.items():
    trip_date = today - datetime.timedelta(days=trip["day_offset"])
    for product in trip["products"]:
        product["path"] = make_product_photo(product["name"], product["color"])
        product["trip_key"] = trip_key
        product["trip_label"] = trip["label"]
        product["trip_date"] = trip_date
        product["trip_hour"] = trip["hour"]
        all_products.append(product)

print(f"Total: {len(all_products)} products across {len(trips)} shopping trips\n")
for trip_key, trip in trips.items():
    n = len(trip["products"])
    stores = set(p["store"] for p in trip["products"])
    print(f"  {trip['label']}: {n} products at {', '.join(stores)}")

# Show a sample from each trip
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (trip_key, trip) in enumerate(trips.items()):
    products = trip["products"]
    for row in range(2):
        if row < len(products):
            p = products[row]
            axes[row, col].imshow(Image.open(p["path"]))
            axes[row, col].set_title(f"${p['price']} {p['desc'][:12]}", fontsize=7)
        axes[row, col].axis("off")
    axes[0, col].set_xlabel(trip["label"][:20], fontsize=8)
plt.suptitle("Products Across 5 Shopping Trips (sample)", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Store Products in On-Device Memory

Embed each product photo with CLIP and store with metadata.

In [ ]:
def embed_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img_tensor = clip_model.image_preprocessor(img).unsqueeze(0)
    with torch.no_grad():
        features = clip_model.clip.encode_image(img_tensor)
        features = features / features.norm(dim=1, keepdim=True)
    return features.squeeze(0).float().numpy()

def embed_text(text):
    tokens = clip.tokenize([text])
    with torch.no_grad():
        features = clip_model.clip.encode_text(tokens)
        features = features / features.norm(dim=1, keepdim=True)
    return features.squeeze(0).float().numpy()

In [ ]:
SHARD_DIR = "./product_scout_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=DIMENSION,
            distance=Distance.Cosine,
        )
    }
)

shard = EdgeShard(SHARD_DIR, config)

print(f"Embedding and storing {len(all_products)} products...")
points = []

for i, product in enumerate(all_products):
    emb = embed_image(product["path"])
    ts = datetime.datetime.combine(product["trip_date"], datetime.time(product["trip_hour"], 0)).timestamp()

    points.append(Point(
        id=i,
        vector={VECTOR_NAME: emb.tolist()},
        payload={
            "description": product["desc"],
            "category": product["category"],
            "store": product["store"],
            "price": product["price"],
            "trip": product["trip_label"],
            "trip_key": product["trip_key"],
            "timestamp": ts,
            "image_path": product["path"],
        }
    ))

shard.update(UpdateOperation.upsert_points(points))
print(f"Stored {len(points)} products across {len(trips)} shopping trips")

## 3. Search Products with Natural Language

"Red shoes I saw" or "something for music" - CLIP matches your text to stored product images.

In [ ]:
def search_products(query, limit=3, payload_filter=None):
    query_emb = embed_text(query)
    kwargs = {
        "query": Query.Nearest(query_emb.tolist(), using=VECTOR_NAME),
        "limit": limit,
        "with_vector": False,
        "with_payload": True,
    }
    if payload_filter:
        kwargs["filter"] = payload_filter
    return shard.query(QueryRequest(**kwargs))

queries = [
    "red shoes",
    "something to listen to music with",
    "a warm jacket",
    "bags and backpacks",
    "a nice watch",
]

for q in queries:
    print(f"\nQ: {q}")
    results = search_products(q)
    for r in results:
        p = r.payload
        print(f"  [{r.score:.3f}] ${p['price']} @ {p['store']} - {p['description']} ({p['trip']})")

## 4. Find Similar Products Across Trips

"I saw headphones today at Target for $249. Where else did I see headphones and how much were they?" This is the core product comparison use case from the Context Hub doc.

In [ ]:
# Today's headphones vs all previous trips
today_headphones = [p for p in all_products if p["trip_key"] == "trip5_today_comparison" and "headphones" in p["desc"].lower()][0]
today_emb = embed_image(today_headphones["path"])

print(f"Today: {today_headphones['desc']} at {today_headphones['store']} for ${today_headphones['price']}")
print(f"\nSimilar items from previous shopping trips:")

results = shard.query(
    QueryRequest(
        query=Query.Nearest(today_emb.tolist(), using=VECTOR_NAME),
        limit=5,
        with_vector=False,
        with_payload=True,
        filter={"must_not": [{"key": "trip_key", "match": {"value": "trip5_today_comparison"}}]}
    )
)

for r in results:
    p = r.payload
    print(f"  [{r.score:.3f}] ${p['price']} @ {p['store']} - {p['description']} ({p['trip']})")

# Same for red sneakers
print("\n" + "=" * 60)
today_sneakers = [p for p in all_products if p["trip_key"] == "trip5_today_comparison" and "sneakers" in p["desc"].lower()][0]
today_emb = embed_image(today_sneakers["path"])

print(f"\nToday: {today_sneakers['desc']} at {today_sneakers['store']} for ${today_sneakers['price']}")
print(f"\nSimilar items from previous trips:")

results = shard.query(
    QueryRequest(
        query=Query.Nearest(today_emb.tolist(), using=VECTOR_NAME),
        limit=5,
        with_vector=False,
        with_payload=True,
        filter={"must_not": [{"key": "trip_key", "match": {"value": "trip5_today_comparison"}}]}
    )
)

for r in results:
    p = r.payload
    print(f"  [{r.score:.3f}] ${p['price']} @ {p['store']} - {p['description']} ({p['trip']})")

## 5. Filter by Price Range

"Show me products under $100" - combine semantic search with price filtering.

In [ ]:
print("Products under $100:")
results = search_products(
    "good deals",
    limit=5,
    payload_filter={
        "must": [{"key": "price", "range": {"lte": 100}}]
    }
)
for r in results:
    p = r.payload
    print(f"  ${p['price']} @ {p['store']} - {p['description']}")

print("\nShoes under $100:")
results = search_products(
    "shoes",
    limit=5,
    payload_filter={
        "must": [
            {"key": "category", "match": {"value": "shoes"}},
            {"key": "price", "range": {"lte": 100}},
        ]
    }
)
for r in results:
    p = r.payload
    print(f"  ${p['price']} @ {p['store']} - {p['description']}")

## 6. Filter by Category

In [ ]:
categories = ["shoes", "clothing", "electronics", "accessories", "bags"]

for cat in categories:
    results = shard.query(
        QueryRequest(
            query=Query.Nearest(embed_text(cat).tolist(), using=VECTOR_NAME),
            limit=10,
            with_vector=False,
            with_payload=True,
            filter={"must": [{"key": "category", "match": {"value": cat}}]}
        )
    )
    items = [f"${r.payload['price']} {r.payload['description']}" for r in results]
    print(f"{cat}: {', '.join(items)}")

## 7. Cleanup

In [ ]:
shard.close()

import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
shutil.rmtree(PRODUCTS_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lab you built a product scout:
- Compiled CLIP for a Snapdragon phone via AI Hub
- Stored 44+ product photos from 5 shopping trips spanning a month
- Searched products with natural language ("red shoes", "something for music")
- Compared today's finds against a month of shopping history using image-to-image similarity
- Filtered by price range, category, and trip

This directly maps to the Context Hub product comparison use case: "I saw headphones at Best Buy for $299. How much are they at Target?"

In the final lab, you'll build a robot memory agent with spatial awareness for navigation.